In [7]:
import pandas as pd

from pathlib import Path

import smtplib, ssl

### Load Scores

In [8]:
exercise_path = Path("course_directory/ads_exercise")
output_path = Path("total_score")

In [9]:
scores = pd.read_csv(exercise_path.joinpath("grades.csv"))
scores.head()

In [10]:
score_long = scores.set_index(["student_id", "assignment"]).sort_index()
score_long

In [11]:
sum_points = score_long["score"].unstack("assignment").sum(axis=1)
sum_points

In [12]:
score_long.xs("AhmedSofan10", level="student_id")

In [13]:
max_score = score_long.xs(score_long.index[0][0], level="student_id")["max_score"].sum() # select the first student
max_score

In [14]:
# set this manually, when the score is not right
max_score = 100.0

In [15]:
percentage = (sum_points / max_score) * 100 
percentage[percentage > 100] = 100
percentage = percentage.sort_values(ascending=False)

In [16]:
percentage.hist(bins=20)

In [17]:
import time

semester = "SoSe_23"

# export csv
timestr = time.strftime("%Y%m%d-%H%M%S")
percentage.to_csv(output_path.joinpath(semester+"-"+timestr+".csv"))

In [18]:
idm_list = pd.read_csv("course_directory/classroom_roster.csv").loc[:,["identifier", "github_username"]]
idm_list

In [19]:
idm_list.set_index("github_username", inplace=True)
idm_list.index.name = "student_id"

In [20]:
idm_list

In [21]:
# join percentage and idm_list

total = pd.DataFrame(percentage, columns=["score_percent"]).join(idm_list, how="inner")
total

In [22]:
# drop all rows with identifier length > 8 

total.drop(total[total['identifier'].str.len() > 8].index, inplace = True)

In [23]:
total

In [16]:
total = total.reset_index().set_index("identifier").drop("student_id", axis=1).sort_index()
total

In [26]:
total.to_csv(output_path.joinpath("2023_total.csv"))

In [18]:
mails = pd.read_csv('studon_export/mails/mails.csv')
mails = mails[["Login", "E-Mail", "First Name", "Last Name"]]
mails

In [19]:
# rename columns
mails = mails.rename({"Login":"identifier", "E-Mail":"email", "First Name":"first_name", "Last Name":"last_name"}, axis=1).set_index("identifier")
mails

In [20]:
email_result = total.join(mails, how="inner")
email_result

In [29]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

import time

In [33]:
# write mail
smtp_server = "smtp.gmail.com"
port = 587  # For starttls
sender_email = "dipsylab@gmail.com"
password = "eestjmljnibjgesu"

# Create a secure SSL context
context = ssl.create_default_context()

# Try to log in to server and send email
try:
    server = smtplib.SMTP(smtp_server,port)
    server.ehlo() # Can be omitted
    server.starttls(context=context) # Secure the connection
    server.ehlo() # Can be omitted
    server.login(sender_email, password)
    
    # send mail here
    for identifier, student in email_result[:5].iterrows():
        message = MIMEMultipart()

        message["From"] = sender_email
        message["To"] = student.email
        message["Subject"] = "ADS Exercise Result"

        # Add body to email
        
        message.attach(MIMEText(f"Hi {student.first_name}, \n\n your total score is {round(student.score_percent, 1)} %.  \n\n Best regards, \n ADS Team", "plain"))
        
        message.attach(MIMEText(f"\n\n This message was automatically generated. Please do not reply to this email.", "plain")

        server.sendmail(sender_email, "luca.abel@fau.de", message.as_string())
        #wait for 1 second

        time.sleep(1)    
    
except Exception as e:
    # Print any error messages to stdout
    print(e)
finally:
    server.quit() 